# Hedge Design Dashboard — SPX tail hedge

Load a book and design changes to it: editor, sizing, strike selection, roll and monetization planning, and stress testing.

In [ ]:
"""Hedge Design Dashboard — SPX tail hedge workbench."""
# Imports
import os
from datetime import UTC as UTC_TZ
from datetime import datetime as dt

import matplotlib.pyplot as plt
from IPython.display import HTML, display

from deltadewa.analysis import (
    PortfolioAnalyzer,
    ScenarioGridCache,
    assess_market_environment,
    build_monetization_plan,
    build_roll_plan,
    build_strike_ladder,
    classify_portfolio_shape,
    compute_crash_convexity,
    decision_matrix,
    entry_timing_tree,
    get_volatility_stats,
    size_hedge,
)
from deltadewa.dashboard import (
    ChangeLogDisplay,
    MonteCarloStalenessWidget,
    StressDashboard,
    VolatilityProfileDisplay,
    start_session,
)
from deltadewa.reporting import PortfolioChangeTracker
from deltadewa.visualization import OptionCharts
from deltadewa.widgets import (
    HedgeHealthDashboard,
    NetHedgeSummary,
    PortfolioWidgets,
)

In [ ]:
# ── Live market data toggle ───────────────────────────────────────
# Set True for live CBOE/FRED data (requires internet:
# cdn.cboe.com, fred.stlouisfed.org). Automatically falls back to
# static if the network is unavailable. Default False = offline-safe.
# See /examples/.env.example for how to set _USE_LIVE=True in your environment.
_USE_LIVE = os.environ.get("_USE_LIVE", "False") == "True"

# Dashboard session: portfolio, market data provider, IPS policy,
# and GlobalAssumptions — all bootstrapped in one call.
ctx = start_session(
    role="design",
    globals_dict=globals(),
    auto_load_default=False,
    use_live_market_data=_USE_LIVE,
)

portfolio = ctx.portfolio
ips_config = ctx.ips_config
dashboard_config = ctx.dashboard_config
market_data = ctx.market_data
global_assumptions = ctx.global_assumptions
reporter = ctx.reporter
portfolio_changelog = ctx.changelog
portfolio_serializer = ctx.serializer
EXPORT_DIR = ctx.export_dir
today = ctx.today

portfolio_widgets = PortfolioWidgets(
    portfolio,
    portfolio_serializer,
    portfolio_changelog,
)

vol_stats = get_volatility_stats(portfolio)

reporter.success("Setup complete.")

## Load / Import book

In [ ]:
# Import Widget and Portfolio Change Tracker

# Create tracker — seeds the baseline snapshot from the current portfolio state
position_tracker = PortfolioChangeTracker(
    portfolio=portfolio,
    logger=portfolio_changelog,
    reporter=reporter,
)

# Import widget — reset the tracker after a successful import so it
# doesn't diff against the pre-import snapshot
import_widget = portfolio_widgets.display_import(
    on_import_success=position_tracker.reset,
)
display(import_widget)

In [ ]:
# Shape guard — check portfolio structure once after load.
_shape = classify_portfolio_shape(portfolio)
_has_book = bool(portfolio.positions) or portfolio.underlying_quantity != 0
if _has_book and not _shape.is_conforming:
    display(
        HTML(
            '<div style="border-left:4px solid #b45309;'
            " background:#fffbeb; padding:10px 14px;"
            " border-radius:4px; max-width:800px;"
            " font-family:sans-serif; font-size:13px;"
            ' color:#78350f;">'
            f"<b>⚠ Portfolio shape:</b> {_shape.notice}"
            "</div>",
        )
    )

## Assumptions (editable scenario inputs)

In [ ]:
display(global_assumptions.display())

## Position Editor

In [ ]:
# Interactive Position Editor using widgets module

# Pass the callback to the position editor widget
position_editor = portfolio_widgets.create_position_editor(
    on_change_callback=position_tracker.as_callback(),
)

display(position_editor)

print("\n💡 All position changes will be automatically logged")

## Net Summary (what I'm building)

In [ ]:
# Hedge Summary
# Crash vol shock is policy: single-sourced from the IPS (never hardcoded).
net_hedge_summary = NetHedgeSummary(
    portfolio,
    crash_vol_shock=(
        ips_config.convexity.crash_vol_shock if ips_config is not None else 0.0
    ),
)
display(net_hedge_summary.display())

## Hedge Health

In [ ]:
# Display Hedge Health Dashboard
# Crash policy (scenario + vol shock bundled), target hedge ratio, and
# vol-regime band are policy: single-sourced from the IPS (never hardcoded).
# Passing the whole convexity object keeps the crash scenario and its vol shock
# tied together — they can't diverge. market_data feeds the vol-regime gauge a
# true percentile when VIX history is available (else honest normalized).
health_dashboard = HedgeHealthDashboard(
    portfolio,
    config=dashboard_config,
    market_data=market_data,
    crash_convexity=(ips_config.convexity if ips_config is not None else None),
    target_delta_ratio_pct=(
        ips_config.triggers.target_delta_ratio_pct
        if ips_config is not None
        else None
    ),
    ips_market_environment=(
        ips_config.market_environment if ips_config is not None else None
    ),
)
dashboard_loader = health_dashboard.display_config_loader()
display(dashboard_loader)
display(health_dashboard.display())
# Update when portfolio changes
health_dashboard.update()

## Decision matrix & entry timing

Decision matrix (handbook Part X): combines whether protection is cheap/expensive
(market environment, live data only) with whether you're under/adequately hedged
(crash convexity vs IPS band) and gains available (monetization) ->
BUY / MAINTAIN / AVOID / MONETIZE.

Re-run after any position change, environment refresh, or monetization update.

In [ ]:
# Decision matrix & entry-timing tree (handbook Part X)
# Reuses _env, _crash, _mon_plan from prior cells if already computed.

_has_book = bool(portfolio.positions) or portfolio.underlying_quantity != 0
if not _has_book or ctx.ips_config is None:
    print(
        "No portfolio or IPS config loaded"
        " — add positions or load an IPS file first.",
    )
else:
    _env_dm = globals().get("_env") or assess_market_environment(
        market_data,
        ctx.ips_config.market_environment,
    )
    _crash_dm = globals().get("_crash") or compute_crash_convexity(
        portfolio,
        ips_convexity=ctx.ips_config.convexity,
    )
    _conv = ctx.ips_config.convexity
    _convexity_now_pct = next(
        (
            r.convexity_pct
            for r in _crash_dm.scenario_rows
            if r.shock_pct == _conv.crash_scenario_pct
        ),
        _crash_dm.payoff_ratio or 0.0,
    )
    _result = decision_matrix(
        _env_dm,
        convexity_now_pct=_convexity_now_pct,
        ips_convexity=_conv,
        monetization_plan=globals().get("_mon_plan"),
    )
    _timing = entry_timing_tree(_env_dm)

    _sep = "-" * 58
    print(_sep)
    print(f"  Verdict  : {_result.verdict}")
    print(f"  Rationale: {_result.rationale}")
    if _result.data_quality_note:
        print()
        print(f"  NOTE: {_result.data_quality_note}")
    print(_sep)
    print(
        f"  Adequacy : {_result.hedge_adequacy}"
        f" ({_convexity_now_pct:.1f}% vs IPS"
        f" {_conv.target_min_pct:.0f}"
        f"-{_conv.target_max_pct:.0f}%)",
    )
    print(f"  Env cost : {_result.cost_verdict}")
    print(
        f"  Gains    : {'Yes' if _result.gains_available else 'No'}",
    )
    print()
    print(f"  Entry timing -- {_timing.recommendation}")
    for _s in _timing.steps:
        _arrow = "->" if _s.proceed else "X "
        print(
            f"    Step {_s.step}"
            f" [{_s.label}={_s.value}]"
            f" {_arrow} {_s.recommendation}",
        )
    if _timing.data_quality_note:
        print(f"  NOTE: {_timing.data_quality_note}")
    print()
    print(
        "Decision matrix (handbook Part X): combines whether"
        " protection is cheap/expensive (market environment, live"
        " data only) with whether you're under/adequately hedged"
        " (crash convexity vs IPS band) and gains available"
        " (monetization) -> BUY / MAINTAIN / AVOID / MONETIZE.",
    )

## Sizing workbench  [NEW — not yet built]

In [ ]:
if not _shape.is_conforming:
    print(f"[shape] {_shape.notice}")
# ---------------------------------------------------------------
# Sizing workbench
# Edit the two variables below, then re-run this cell.
# ---------------------------------------------------------------
_candidate_pct_otm = 20.0  # % OTM  (20 -> strike = spot x 0.80)
_candidate_maturity_years = 0.5  # years to expiry  (0.5 -> ~6 months)

print(
    "Sizing = (crash loss beyond drawdown tolerance) /"
    " (per-contract crash payoff), capped by the annual carry budget."
    " Payoff is the candidate repriced at the crash state (crash spot +"
    " vol shock); the intrinsic floor is a conservative lower bound."
    " Convexity shown vs the IPS target band.",
)
print()

# Sizing needs an underlying position (the book notional it sizes against);
# without one size_hedge raises rather than fabricate a zero-notional result.
if portfolio.underlying_quantity <= 0 or ctx.ips_config is None:
    print(
        "Sizing needs an underlying position and an IPS config"
        " — set underlying_quantity and load an IPS file first.",
    )
else:
    _r = size_hedge(
        portfolio,
        ctx.ips_config,
        candidate_pct_otm=_candidate_pct_otm,
        candidate_maturity_years=_candidate_maturity_years,
    )
    _strike = portfolio.spot_price * (1.0 - _candidate_pct_otm / 100.0)
    _conv = ctx.ips_config.convexity
    _rows = [
        (
            "Required crash offset",
            f"${_r.required_crash_offset:,.0f}",
        ),
        (
            "Candidate put",
            (
                f"${_strike:,.0f}"
                f" ({_candidate_pct_otm:.1f}% OTM,"
                f" {_candidate_maturity_years:.2f} yr)"
            ),
        ),
        (
            "Per-contract crash payoff",
            f"${_r.per_contract_payoff:,.0f}",
        ),
        (
            "Per-contract intrinsic floor",
            f"${_r.per_contract_intrinsic_floor:,.0f}",
        ),
        ("Contracts needed", f"{_r.contracts_needed:,d}"),
        (
            "Implied annual carry",
            f"${_r.implied_annual_carry:,.0f}",
        ),
        ("Carry budget", f"${_r.carry_budget:,.0f}"),
        (
            "Within budget?",
            (
                "YES"
                if _r.within_budget
                else (f"NO  max affordable: {_r.max_affordable_contracts:,d}")
            ),
        ),
        ("Carry headroom", f"${_r.carry_headroom:,.0f}"),
        (
            "Achieved convexity",
            f"{_r.achieved_convexity_pct:.1f}%",
        ),
        (
            "IPS convexity band",
            (f"{_conv.target_min_pct:.1f}%-{_conv.target_max_pct:.1f}%"),
        ),
        (
            "Meets convexity target?",
            "YES" if _r.meets_convexity_target else "NO",
        ),
    ]
    _w = max(len(lbl) for lbl, _ in _rows) + 2
    _sep = "-" * (_w + 24)
    print(_sep)
    for lbl, val in _rows:
        print(f"  {lbl:<{_w}}{val}")
    print(_sep)

## Strike ladder builder

In [ ]:
# ---------------------------------------------------------------
# Strike ladder builder
# Edit the lists below, then re-run this cell.
# ---------------------------------------------------------------
_target_deltas = [0.05, 0.10, 0.15]  # put-delta magnitudes
_maturities_years = [0.25, 0.5, 1.0]  # years to expiry

# The ladder sizes each rung against the book notional, so it needs an
# underlying position; without one build_strike_ladder raises.
if portfolio.underlying_quantity <= 0 or ctx.ips_config is None:
    print(
        "Strike ladder needs an underlying position and an IPS config"
        " — set underlying_quantity and load an IPS file first.",
    )
else:
    _ladder_result = build_strike_ladder(
        portfolio,
        ctx.ips_config,
        target_deltas=_target_deltas,
        maturities_years=_maturities_years,
    )
    _ladder = _ladder_result.rungs
    if not _ladder:
        print("No solvable (delta, maturity) combinations found.")
    else:
        _conv = ctx.ips_config.convexity
        _bgt = _ladder[0].carry_budget
        print(
            "Strike ladder"
            f" | crash {_conv.crash_scenario_pct:.0f}%"
            f" | band {_conv.target_min_pct:.0f}"
            f"-{_conv.target_max_pct:.0f}%"
            f" | carry budget ${_bgt:,.0f}/yr",
        )
        print()
        _hdr = (
            "    "
            f"{'delta':>5}  {'mat':>5}  {'strike':>7}"
            f"  {'%OTM':>6}  {'prem $':>8}  {'payoff $':>9}"
            f"  {'floor $':>9}"
            f"  {'carry $/yr':>10}  {'cts':>4}"
            f"  {'impl carry $':>13}  {'conv %':>7}"
        )
        print(_hdr)
        print("-" * len(_hdr))
        for _r in _ladder:
            _ok = "OK" if _r.meets_target_within_budget else "  "
            _m = _r.metrics
            print(
                f"{_ok:2}  {_r.target_delta:>5.2f}"
                f"  {_r.maturity_years:>5.2f}"
                f"  {_m.strike:>7,.0f}"
                f"  {_m.pct_otm:>5.1f}%"
                f"  {_m.premium:>8,.0f}"
                f"  {_m.per_contract_payoff:>9,.0f}"
                f"  {_m.per_contract_intrinsic_floor:>9,.0f}"
                f"  {_m.per_contract_carry:>10,.0f}"
                f"  {_r.contracts_needed:>4}"
                f"  {_r.implied_annual_carry:>13,.0f}"
                f"  {_r.achieved_convexity_pct:>6.1f}%",
            )
        print()
        print(
            "OK = meets IPS convexity target within carry budget.",
        )
        print()
        print(
            "Each rung is a candidate put selected by target delta."
            " Compare cost (premium, carry) against protection"
            " (crash payoff, convexity) to pick a structure;"
            " marked rungs meet the IPS convexity target within"
            " the carry budget.",
        )

    # Surface unsolvable cells explicitly rather than dropping them silently.
    if _ladder_result.unsolvable:
        print()
        print("Unsolvable rungs (no OTM strike for the requested delta):")
        for _u in _ladder_result.unsolvable:
            print(
                f"  delta {_u.target_delta:>4.2f}"
                f" @ {_u.maturity_years:>4.2f}y — {_u.reason}",
            )

## Roll planner

In [ ]:
# ── Roll Planner ──────────────────────────────────────────────────
# Edit these two variables to choose the target-strike basis:
_roll_target_basis = "entry_otm"  # "entry_otm" or "delta"
_roll_target_delta = 0.10  # desired put-delta (used when basis="delta")

if portfolio is not None and ips_config is not None:
    plan = build_roll_plan(
        portfolio,
        ips_config,
        target_basis=_roll_target_basis,
        target_delta=_roll_target_delta,
    )
    if not plan:
        print("No long puts in portfolio.")
    else:
        _conv = ips_config.convexity
        band = f"{_conv.target_min_pct:.0f}-{_conv.target_max_pct:.0f}%"
        _hdr = (
            f"{'Leg':<20} {'Verdict':<9} {'Action':<10}"
            f" {'Target strike (basis)':<24} {'Roll cost':>10}"
            f"  {'Conv. now':>9}  {'IPS band':>8}"
        )
        print(_hdr)
        print("-" * len(_hdr))
        for _rec in plan:
            _pos = _rec.position
            _leg = f"${_pos.option.strike_price:.0f} x{_pos.quantity}"
            _strike_label = (
                f"${_rec.target_strike:.0f} ({_rec.target_basis})"
                if _rec.target_strike is not None
                else f"n/a ({_rec.target_basis})"
            )
            _cost_label = (
                f"${_rec.roll_up_cost:,.0f}"
                if _rec.roll_up_cost is not None
                else "n/a"
            )
            print(
                f"{_leg:<20} {_rec.verdict:<9} {_rec.action:<10}"
                f" {_strike_label:<24} {_cost_label:>10}"
                f"  {_rec.convexity_now_pct:>8.1f}%  {band:>8}",
            )
            print(f"  -> {_rec.rationale}")
        print()
        print(
            "Roll rules (handbook Part VII): roll on the time trigger"
            " (maturity within roll_time_months), re-strike after a"
            " rally, or on strike drift; but DELAY a mechanical roll"
            " while convexity still meets the IPS target to keep"
            " collecting gamma before paying theta.",
        )
else:
    print("No portfolio or IPS config loaded.")

## Monetization planner  [NEW — not yet built]

In [ ]:
# ---------------------------------------------------------------
# Monetization planner  (handbook Part VIII §3729)
# Re-run after any position or gain change.
# ---------------------------------------------------------------
_has_book = bool(portfolio.positions) or portfolio.underlying_quantity != 0
if not _has_book or ctx.ips_config is None:
    print(
        "No portfolio or IPS config loaded — add positions or"
        " load an IPS file first.",
    )
else:
    _mon_plan = build_monetization_plan(
        portfolio,
        ctx.ips_config,
        market_env=globals().get("_env"),
    )
    if _mon_plan.current_gain_pct is None:
        _gain_str = "unknown (entry_premium not set on all long puts)"
    else:
        _gain_str = f"{_mon_plan.current_gain_pct:+.1f}%"
    print(
        f"Current hedge gain : {_gain_str}  (basis: {_mon_plan.gain_basis})",
    )
    print()
    _hdr = (
        f"  {'Step':>4}  {'Trigger (gain ≥)':>16}  {'Sell %':>6}  {'Status':>9}"
    )
    print(_hdr)
    print("-" * len(_hdr))
    for _i, _s in enumerate(_mon_plan.steps, 1):
        _status = "TRIGGERED" if _s.triggered else "—"
        print(
            f"  {_i:>4}  {_s.gain_pct:>+15.1f}%"
            f"  {_s.sell_pct:>5.1f}%"
            f"  {_status:>9}",
        )
    print()
    print(
        f"Recommended cumulative sell :"
        f" {_mon_plan.recommended_cumulative_sell_pct:.1f}%",
    )
    print(
        f"Value to harvest now        : ${_mon_plan.value_to_harvest:,.0f}",
    )
    print(
        f"Remaining IPS capacity      :"
        f" {_mon_plan.remaining_sell_capacity:.1f}%",
    )
    if _mon_plan.vol_spike_context:
        print()
        print(f"  NOTE: {_mon_plan.vol_spike_context}")
    print()
    print(
        "Monetization (handbook Part VIII): as the hedge gains,"
        " the IPS schedule says what fraction to harvest. Shown"
        " is the RECOMMENDED cumulative sell at the current gain"
        " (not realized — executed sells aren't tracked yet).",
    )

## Monte Carlo (run for risk/reward)

In [ ]:
# Monte Carlo Simulation - Run Now For Use in Downstream Analysis
#
# This cell runs the Monte Carlo simulation once and stores results on the
# portfolio object. All downstream widgets and visualizations will use
# these cached results for consistency and efficiency.

num_simulations = int(global_assumptions.monte_carlo_num_sims.value)
include_underlying = global_assumptions.monte_carlo_inc_ul.value

reporter.subheader(
    "Monte Carlo Simulation - Run Now For Use in Downstream Analysis",
)
if len(portfolio.positions) > 0:
    print(
        f"Running {num_simulations:,} simulations with underlying: "
        f"{include_underlying}...",
    )
    mc_results = portfolio.run_monte_carlo_simulation(
        num_simulations=num_simulations,
        include_underlying=include_underlying,
    )

    # Mark Monte Carlo results as fresh (not stale)
    portfolio.monte_carlo_stale = False
    portfolio.monte_carlo_timestamp = dt.now(tz=UTC_TZ)

    reporter.success(
        f"Simulation complete: "
        f"{mc_results['num_simulations']:,} valid scenarios",
    )
    reporter.info(
        f"We have {'stale' if portfolio.monte_carlo_stale else 'fresh'} results"
        f" as of {portfolio.monte_carlo_timestamp}",
    )
else:
    reporter.error("No positions - Monte Carlo simulation skipped")
    mc_results = None

reporter.divider()

In [ ]:
# Initialize Scenario Grid Cache and Stress Dashboard

# This caches expensive scenario calculations for better performance
scenario_cache = ScenarioGridCache(max_size=128)
reporter.success("Scenario cache initialized")

analyzer = PortfolioAnalyzer(portfolio)
reporter.success("Portfolio analyzer ready")

stress_dashboard = StressDashboard(
    portfolio=portfolio,
    analyzer=analyzer,
    cache=scenario_cache,
    global_assumptions=global_assumptions,
    reporter=reporter,
)
reporter.success("StressDashboard ready")

## P&L of proposed structure

In [ ]:
# P&L Distribution Chart with Annotated Key Metrics
if len(portfolio.positions) > 0:
    charts = OptionCharts(portfolio)

    # Generate the chart
    fig = charts.plot_pnl_distribution_with_metrics(
        spot_range_pct=max(25.0, global_assumptions.spot_shock_pct.value * 100),
        num_points=500,  # High resolution for smooth curve
        include_underlying=True,  # Include underlying position
        show_probability_overlay=True,  # Show probability density function
    )

    plt.show()

else:
    print("No positions to analyze. Add positions in BUILD mode.")

## Stress: Time×Price

In [ ]:
# Time vs Price Heatmap Analysis
if len(portfolio.positions) > 0:
    time_heatmap_widget = stress_dashboard.create_time_heatmap(metric="pnl")
    display(time_heatmap_widget)
else:
    reporter.error(
        "No positions to analyze. Add positions in BUILD mode first.",
    )

## Stress: Vol×Price

In [ ]:
# Interactive Stress Test Heatmap (Spot x Volatility)
if len(portfolio.positions) > 0:
    spot_vol_widget = stress_dashboard.create_spot_vol_heatmap(
        metric="pnl",
        days_forward=0,
    )
    display(spot_vol_widget)
else:
    reporter.error(
        "No positions to analyze. Add positions in BUILD mode first.",
    )

## Risk / Reward (Monte Carlo)

In [ ]:
# Monte Carlo Staleness Check & Re-run Widget
mc_widget = MonteCarloStalenessWidget(
    portfolio,
    num_simulations,
    include_underlying,
    reporter,
)
is_stale = mc_widget.check_and_warn()

reporter.header(" Monte Carlo P&L Distribution Analysis")

mc_results = portfolio.monte_carlo_results
if mc_results is None:
    reporter.warning("No Monte Carlo results found. Running simulation now...")
    mc_results = portfolio.run_monte_carlo_simulation(
        num_simulations=num_simulations,
        include_underlying=include_underlying,
    )
    portfolio.monte_carlo_stale = False
    portfolio.monte_carlo_timestamp = dt.now(tz=UTC_TZ)
else:
    if not is_stale:
        reporter.success(
            "Using cached Monte Carlo results from earlier simulation",
        )
    else:
        reporter.warning("Using STALE cached results (portfolio was modified)")

# Delegate all rendering to StressDashboard
stress_dashboard.display_risk_reward_summary(mc_results)

## Volatility Profile

In [ ]:
# Display Portfolio Volatility Profile
VolatilityProfileDisplay(portfolio, reporter).display(vol_stats)

## Session Change Log

In [ ]:
# Portfolio Change Log Display
ChangeLogDisplay(portfolio_changelog, reporter).display()

## Export working portfolio

In [ ]:
# Final Export Widget
final_export_widget = portfolio_widgets.display_export()
display(final_export_widget)

reporter.header("SESSION COMPLETE")
print("Remember to export your portfolio to save your work!")
print("Use the widget above to export in JSON, CSV, or YAML format.")
reporter.divider()